# 🚀 Globex Corp RAG — Desarrollo en Google Colab

Notebook completo para **desarrollar, probar y mostrar** el agente RAG
sin salir de Colab. Incluye URL pública temporal vía `pyngrok` para ver
la interfaz de Streamlit real desde el navegador.

**Flujo:**
1. Instalar dependencias
2. Crear los archivos del proyecto en Colab
3. Configurar la API key (usando Secrets de Colab)
4. Generar los documentos PDF
5. Probar la lógica RAG en celdas
6. Levantar Streamlit con URL pública (para evidencia del challenge)


## Celda 1 — Instalar dependencias

In [ ]:
%pip install -q streamlit pyngrok langchain langchain-community \
    langchain-groq faiss-cpu sentence-transformers pypdf \
    reportlab python-dotenv

print("✅ Dependencias instaladas")


## Celda 2 — Configurar API Keys

Guarda tus claves en **Colab Secrets** (🔑 panel izquierdo) con estos nombres:
- `GROQ_API_KEY` → tu clave de https://console.groq.com/keys
- `NGROK_AUTHTOKEN` → tu token de https://dashboard.ngrok.com/get-started/your-authtoken

Ambas son gratuitas. Ngrok es necesario solo para la celda de Streamlit (paso 6).


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"]     = userdata.get("GROQ_API_KEY")
os.environ["NGROK_AUTHTOKEN"]  = userdata.get("NGROK_AUTHTOKEN")

# Verificar que se cargaron correctamente
groq_ok  = bool(os.environ.get("GROQ_API_KEY"))
ngrok_ok = bool(os.environ.get("NGROK_AUTHTOKEN"))

print(f"GROQ_API_KEY:    {'✅ configurada' if groq_ok  else '❌ falta — ve a Secrets'}")
print(f"NGROK_AUTHTOKEN: {'✅ configurado' if ngrok_ok else '❌ falta — ve a Secrets (solo necesario para Streamlit)'}")


## Celda 3 — Crear los archivos del proyecto

Crea la estructura de carpetas y escribe los scripts directamente en Colab.
Si ya subiste los archivos manualmente desde el panel 📁, puedes saltarte esta celda.


In [ ]:
import os

os.makedirs("documentos", exist_ok=True)

# ── generar_pdf_dummy.py (manual de RH) ──────────────────────────────
codigo_rh = '''
import os
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
    PageBreak, ListFlowable, ListItem, Table, TableStyle)

CARPETA_SALIDA = "documentos"
os.makedirs(CARPETA_SALIDA, exist_ok=True)
estilos = getSampleStyleSheet()
E_TITULO  = ParagraphStyle("T", parent=estilos["Title"], fontSize=22, alignment=TA_CENTER, textColor=colors.HexColor("#0B3D91"))
E_SUB     = ParagraphStyle("S", parent=estilos["Normal"], fontSize=12, alignment=TA_CENTER, textColor=colors.HexColor("#555555"))
E_SEC     = ParagraphStyle("H1", parent=estilos["Heading1"], fontSize=15, textColor=colors.HexColor("#0B3D91"), spaceBefore=14)
E_SUBSEC  = ParagraphStyle("H2", parent=estilos["Heading2"], fontSize=12, textColor=colors.HexColor("#1F4E8C"), spaceBefore=8)
E_CUERPO  = ParagraphStyle("C", parent=estilos["Normal"], fontSize=10.5, leading=15, alignment=TA_JUSTIFY, spaceAfter=7)
E_ITEM    = ParagraphStyle("I", parent=E_CUERPO, leftIndent=12)

def lista(items): return ListFlowable([ListItem(Paragraph(i, E_ITEM)) for i in items], bulletType="bullet")
def tabla(datos, widths):
    t = Table(datos, colWidths=widths)
    t.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.HexColor("#0B3D91")),("TEXTCOLOR",(0,0),(-1,0),colors.white),
        ("FONTSIZE",(0,0),(-1,-1),9),("GRID",(0,0),(-1,-1),0.5,colors.grey),
        ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.whitesmoke,colors.white]),
        ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
    return t

c = [Spacer(1,1.2*inch), Paragraph("GLOBEX CORP", E_TITULO),
     Paragraph("Manual de Políticas Internas — Recursos Humanos", E_SUB), PageBreak(),
     Paragraph("1. Política de Vacaciones y Licencias", E_SEC),
     Paragraph("1.1 Días según antigüedad", E_SUBSEC),
     tabla([["Antigüedad","Días hábiles/año"],["Menos de 1 año","12 (proporcional)"],
            ["1 a 4 años","15"],["5 a 9 años","20"],["10 a 14 años","25"],["15+ años","30"]],
           [2.5*inch, 3.0*inch]),
     Spacer(1,8),
     Paragraph("1.2 Proceso de solicitud", E_SUBSEC),
     Paragraph("Las solicitudes se registran en el portal Globex People con 15 días de anticipación mínima. El jefe directo tiene 3 días hábiles para aprobar o rechazar.", E_CUERPO),
     Paragraph("2. Viáticos, Reembolsos y Gastos de Representación", E_SEC),
     tabla([["Concepto","Nacional (USD/día)","Internacional (USD/día)"],
            ["Alimentación","35","60"],["Transporte local","20","40"],
            ["Hospedaje","80","150"]],[2.2*inch, 1.8*inch, 2.0*inch]),
     Spacer(1,8),
     Paragraph("3. Modalidad de Trabajo Remoto / Home Office", E_SEC),
     Paragraph("Se otorga un apoyo mensual de USD 25 para internet y USD 10 para electricidad a colaboradores en modalidad remota total o híbrida.", E_CUERPO),
     lista(["Remoto total: hasta 5 días/semana.","Híbrido estándar: 3 días remoto, 2 en oficina.","Remoto ocasional: hasta 2 días al mes."]),
     Paragraph("4. Código de Ética y Canal de Denuncias Internas", E_SEC),
     Paragraph("Globex Corp cuenta con la Línea Ética Globex, un canal confidencial y anónimo disponible 24/7 para reportar conductas contrarias al Código de Ética.", E_CUERPO),
]
ruta = os.path.join(CARPETA_SALIDA, "Manual_Politicas_GlobexCorp.pdf")
doc = SimpleDocTemplate(ruta, pagesize=LETTER, topMargin=0.9*inch, bottomMargin=0.9*inch, leftMargin=0.9*inch, rightMargin=0.9*inch)
doc.build(c)
print("✅ Generado:", ruta)
'''

with open("generar_pdf_dummy.py", "w") as f:
    f.write(codigo_rh)

print("✅ Archivos del proyecto creados en Colab.")
print("   Si tienes los archivos completos del repo, súbelos desde el panel 📁 y omite esta celda.")


## Celda 4 — Generar los documentos PDF

In [ ]:
import subprocess, sys

# Si subiste los scripts del repo completo, esto corre los 2 generadores.
# Si solo creaste generar_pdf_dummy.py en la celda anterior, solo corre ese.
for script in ["generar_pdf_dummy.py", "generar_documentos_ecommerce.py"]:
    if os.path.exists(script):
        result = subprocess.run([sys.executable, script], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print("⚠️ Error en", script)
            print(result.stderr)
    else:
        print(f"⚠️ {script} no encontrado — sube el archivo o ejecuta la celda 3.")

import os
print("\nDocumentos disponibles:", os.listdir("documentos"))


## Celda 5 — Probar la lógica RAG directamente

Carga los PDFs, crea el índice FAISS y prueba el agente sin necesitar Streamlit.
Cambia `pregunta` y vuelve a correr esta celda para iterar rápido.


In [ ]:
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# ── Cargar y trocear documentos ───────────────────────────────────────
pdfs = glob.glob("documentos/*.pdf")
print(f"PDFs encontrados: {len(pdfs)}")

docs = []
for p in pdfs:
    docs.extend(PyPDFLoader(p).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=300)
chunks = splitter.split_documents(docs)
print(f"Fragmentos totales: {len(chunks)}")

# ── Crear índice vectorial ─────────────────────────────────────────────
print("\nCreando embeddings y vectorstore (1-2 min)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("✅ Vectorstore creado.")

# ── Armar la cadena RAG ────────────────────────────────────────────────
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

prompt_qa = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Eres el asistente virtual de Globex Corp. Responde SIEMPRE en "
        "español, basándote únicamente en el siguiente contexto. "
        "Si no está en el contexto, dilo explícitamente.\n\n"
        "Contexto:\n{context}\n\nPregunta: {question}\nRespuesta:"
    ),
)

memoria = ConversationBufferMemory(
    memory_key="chat_history", return_messages=True, output_key="answer"
)

cadena = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    memory=memoria,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": prompt_qa},
)
print("✅ Agente RAG listo.\n")

# ── Hacer una pregunta ─────────────────────────────────────────────────
pregunta = "¿Cuántos días de vacaciones me corresponden con 6 años de antigüedad?"
# Cambia la pregunta y vuelve a correr SOLO esta celda (el vectorstore ya está en memoria)

resultado = cadena.invoke({"question": pregunta})
print("PREGUNTA:", pregunta)
print("\nRESPUESTA:")
print(resultado["answer"])
print("\n📎 Fuentes:")
for doc in resultado["source_documents"]:
    print(f"  - {os.path.basename(doc.metadata.get('source',''))} | pág {doc.metadata.get('page',0)+1}")


## Celda 6 — Chat interactivo por consola

Útil para probar varias preguntas seguidas y verificar que la memoria
conversacional funciona (ej: pregunta algo, luego escribe "¿y con 10 años?").

Escribe `salir` para terminar.


In [ ]:
# Asegúrate de haber corrido la Celda 5 primero (construye 'cadena')
while True:
    p = input("Tu pregunta ('salir' para terminar): ").strip()
    if p.lower() == "salir":
        break
    if not p:
        continue
    r = cadena.invoke({"question": p})
    print("\n🤖", r["answer"])
    fuentes = list({os.path.basename(d.metadata.get("source","")) for d in r["source_documents"]})
    print("📎 Fuentes:", fuentes, "\n")


## Celda 7 — Levantar Streamlit con URL pública (ngrok)

Esta celda levanta la interfaz visual completa y te da una URL pública
temporal que puedes abrir en cualquier navegador o compartir para mostrar
que la app funciona. Úsala para **tomar la captura de pantalla del README**.

⚠️ Requiere:
- Tener el archivo `app.py` subido al entorno de Colab (del repo del proyecto).
- Haber configurado `NGROK_AUTHTOKEN` en Secrets (celda 2).
- La URL expira cuando se cierra la sesión de Colab o se detiene la celda.


In [ ]:
import subprocess, time, os
from pyngrok import ngrok, conf

# Verificar que app.py existe
if not os.path.exists("app.py"):
    print("❌ No se encontró app.py.")
    print("   Sube el archivo desde el panel de archivos 📁 y vuelve a correr esta celda.")
else:
    # Escribir el .env para que app.py pueda leer la API key
    with open(".env", "w") as f:
        f.write(f"GROQ_API_KEY={os.environ.get('GROQ_API_KEY', '')}\n")

    # Configurar y autenticar ngrok
    conf.get_default().auth_token = os.environ["NGROK_AUTHTOKEN"]

    # Matar procesos de Streamlit previos si hubiera alguno
    os.system("pkill -f streamlit 2>/dev/null || true")
    time.sleep(1)

    # Lanzar Streamlit en segundo plano
    proceso = subprocess.Popen(
        ["streamlit", "run", "app.py",
         "--server.port=8501",
         "--server.headless=true",
         "--server.address=0.0.0.0"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(4)  # Esperar a que Streamlit inicie

    # Abrir el túnel ngrok
    tunel = ngrok.connect(8501)
    url_publica = tunel.public_url

    print("=" * 60)
    print("✅ ¡Streamlit está corriendo!")
    print(f"\n🌐 URL pública: {url_publica}")
    print("\nAbre ese link en tu navegador. Funcionará mientras")
    print("esta celda esté ejecutándose (no la interrumpas).")
    print("\nPara detenerlo, interrumpe la celda (botón ⏹️)")
    print("=" * 60)

    # Mantener el túnel abierto
    try:
        proceso.wait()
    except KeyboardInterrupt:
        proceso.terminate()
        ngrok.disconnect(url_publica)
        print("\n🛑 Streamlit y ngrok detenidos.")


## Flujo de trabajo recomendado mientras desarrollas

```
Cambias el prompt o el chunk_size
        ↓
Corres celda 5 (reconstruye el pipeline con los cambios)
        ↓
Pruebas preguntas con celda 6 (chat interactivo)
        ↓
¿Funciona bien? → Corres celda 7 para ver la interfaz visual
        ↓
Tomas captura → la pegas en el README
        ↓
git commit + git push → GitHub
        ↓
Deploy final en OCI (una sola vez, cuando todo esté listo)
```

**Parámetros que puedes ajustar en celda 5 para experimentar:**
- `chunk_size`: valores entre 800–1500 (más grande = más contexto por chunk, pero menos precisión de búsqueda)
- `chunk_overlap`: entre 150–400 (más overlap = menos riesgo de cortar una respuesta a la mitad)
- `k=4` en el retriever: cuántos chunks se le pasan al LLM (más = más contexto, pero también más ruido)
- `temperature=0.2` en el LLM: más bajo = más determinista, más alto = más creativo (para un RAG, 0–0.3 es ideal)
